In [22]:
import warnings 
warnings.filterwarnings("ignore")
import langchain_community
from langchain_community.document_loaders import PyPDFLoader
loder = PyPDFLoader('C:\\Users\\SAYAN METE\\OneDrive\\Documents\\Retrival Argumented Generation\Introduction to Machine Learning with Python ( PDFDrive.com )-min.pdf')
pages = loder.load()


In [23]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
spliter = RecursiveCharacterTextSplitter(chunk_size = 4000,chunk_overlap = 750)
texts = spliter.split_documents(pages)

chunks = []
for i in texts:
    chunks.append(i.page_content)
metadata = []
for doc in texts:
    metadata.append(doc.metadata)


In [24]:
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
embedding_function = SentenceTransformerEmbeddingFunction()
client = chromadb.PersistentClient(path=".\hybrid-rag2")
collection = client.get_or_create_collection(name="Collection",embedding_function=embedding_function)

try:
    if collection.count()==0:
        collection.add(
            documents = chunks,
            ids = [str(i) for i in range(len(chunks))],
            metadatas=metadata
        )

except Exception as e:
    print(str(e))


from rank_bm25 import BM25Okapi

def token_create(i):
    i = i.lower()
    i = i.split()
    return i 
token = [token_create(i) for i in chunks]
token_corpus = BM25Okapi(token)


In [25]:
def retrival(query:str)->str:
    query_lower = query.lower()
    response =  collection.query(query_texts=[query_lower],n_results=5)
    document = response['documents'][0]
    distance = response['distances'][0]

    thresold = 1.6
    near_chunks = []
    for i , j in zip(distance,document):
        if thresold>i:
            near_chunks.append(j)

    score = token_corpus.get_scores(token_create(query_lower))

    def near_index_find(score,k = 10):
        index = list(enumerate(score))
        index_sortde = sorted(index,key=lambda x:x[1],reverse=True)
        return [inx for inx , sc in index_sortde[:k]]
    get_index = near_index_find(score,k = 10)

    index_to_chunks = []
    for i in get_index:
        index_to_chunks.append(chunks[i])

    rrf_item = {}

    for rank , doc in enumerate(near_chunks):
        rrf_item[doc] = rrf_item.get(doc,0)+1/(rank+60)

    for rank , doc in enumerate(index_to_chunks):
        rrf_item[doc] = rrf_item.get(doc,0)+1/(rank+60)

    merge = sorted(rrf_item.items(),key=lambda x:x[1],reverse=True)

    top_doc = []

    for doc , _ in merge[:5]:
        top_doc.append(doc)
    if not top_doc:
        return "NOT RELATED CONTENT"
    return "\n\n".join(top_doc)



In [26]:
from langchain_groq import ChatGroq
import os 
from dotenv import load_dotenv
load_dotenv()
api = os.getenv('GROQ_API_KEY')

groq_llm_model = ChatGroq(model="openai/gpt-oss-120b",api_key=api)

question = input('ask your question ??')
content = retrival(question)

prompt = """
You are a knowledgeable Machine Learning assistant built on the book 
    "Introduction to Machine Learning with Python". Your job is to answer questions 
    strictly based on the retrieved context provided from the book.

    Guidelines:
    1. Use ONLY the information present in the retrieved context to answer the question.
    2. If the retrieved context does not contain enough information to answer the 
    question, clearly say: "I couldn't find relevant information in the book to 
    answer this question." Do NOT make up or hallucinate an answer.
    3. When explaining ML concepts, keep the explanation clear, beginner-friendly, 
    and technically accurate — use simple language and examples where possible.
    4. If code snippets are present in the retrieved context, include them properly 
    formatted in your answer.
    5. Cite which part/chapter/section of the book the answer is coming from, if 
    that information is available in the retrieved context.
    6. Keep answers concise and well-structured. Use bullet points or numbered 
    steps when explaining a process (e.g., an algorithm's steps).
    7. Do not answer questions unrelated to Machine Learning or outside the scope 
    of the retrieved context.

    Context:{content}

    question:{question}


"""

final_prompt = prompt.format(content = content , question = question)

print(groq_llm_model.invoke(final_prompt).content)


**Supervised learning** is a type of machine‑learning where the algorithm learns from **input‑output pairs** (also called *training examples*).  

- **Goal:** Build a model that can predict the correct output for new, unseen inputs.  
- **How it works:**  
  1. A *teacher* (the dataset) provides examples consisting of an input (features) and the desired output (label).  
  2. The algorithm fits a function that maps inputs to outputs using this labeled data.  
  3. After training, the model can automatically generate predictions for future inputs without further human help.  

The book describes supervised learning as the approach used for tasks such as classifying iris flowers, detecting spam email, and many other problems where labeled data are available (see **Chapter 2 – Supervised Learning**).
